# Comparación universitaria: VGG-16 y YOLO26n-cls

Autor: Yefrey Sneyder Sanchez Alba. Asignatura: Inteligencia artificial.

[Repositorio del proyecto](https://github.com/Sneyderx/comparacion-escoliosis-vgg16-yolo)

Este notebook ejecuta ambos clasificadores con el protocolo conservado en el repositorio. Los resultados entregados están en `results/experiment/`.

Antes de ejecutar: activar una GPU y subir `scoliosis_depurado.zip` a la raíz de Mi unidad en Google Drive. La primera celda clona el código automáticamente.

Si solo quieres revisar los resultados existentes, consulta el README y el artículo; no es necesario volver a entrenar.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, os, subprocess, sys, json

drive.mount('/content/drive')
PROJECT = Path('/content/proyecto_escoliosis')
if not (PROJECT / 'config.json').is_file():
    assert not PROJECT.exists(), 'La carpeta de destino existe sin config.json; usa una sesión limpia.'
    subprocess.run(['git', 'clone', 'https://github.com/Sneyderx/comparacion-escoliosis-vgg16-yolo.git', str(PROJECT)], check=True)
os.chdir(PROJECT)

def extract_checked(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for item in z.infolist():
            target = (destination / item.filename).resolve()
            assert target == destination or destination in target.parents, 'Ruta inválida en ZIP'
        z.extractall(destination)

print('Proyecto:', PROJECT)


In [ ]:
%pip install -q -r requirements.txt
import torch, torchvision, ultralytics
print('PyTorch:', torch.__version__, 'TorchVision:', torchvision.__version__, 'Ultralytics:', ultralytics.__version__)
assert torch.cuda.is_available(), 'Activa GPU y vuelve a ejecutar las celdas.'
print('GPU:', torch.cuda.get_device_name(0))
RESULTS = Path('/content/drive/MyDrive/escoliosis_resultados_v1')
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / 'installed_versions.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Preparar datos

Si el ZIP está en otra carpeta de Drive, cambia `DATASET_ZIP`. La extracción se hace en el disco temporal de Colab para acelerar lectura. El registro de clases se vuelve a comprobar y no se modifica la partición.

In [ ]:
DATASET_ZIP = Path('/content/drive/MyDrive/scoliosis_depurado.zip')
assert DATASET_ZIP.is_file(), f'No se encuentra {DATASET_ZIP}; sube el dataset a Drive o ajusta esta ruta.'
DATASET = Path('/content/scoliosis_depurado')
if not (DATASET / 'manifest.csv').is_file():
    extract_checked(DATASET_ZIP, '/content')
subprocess.run([sys.executable, 'data/prepare_classification.py', '--dataset', str(DATASET), '--output', 'data'], check=True)
print(json.loads(Path('data/classification_audit.json').read_text()))


## Configuración fija y prueba corta

La prueba corta usa cuatro imágenes de entrenamiento y cuatro de validación por modelo, una época, 64×64 y pesos aleatorios. Comprueba entrenamiento, cabeza de cuatro clases y selección de checkpoint. **No es un experimento válido ni evalúa prueba.** Si falla, detenerse y compartir el error.

Si se necesita cambiar el lote por memoria, usar el mismo para ambos y fijarlo antes de los entrenamientos completos. Cambiar también la carpeta `RESULTS` para no mezclar protocolos.

In [ ]:
cfg = json.loads(Path('config.json').read_text(encoding='utf-8-sig'))
print(json.dumps(cfg, indent=2))
subprocess.run([sys.executable, 'src/run_all.py', '--stage', 'smoke', '--dataset', str(DATASET), '--results', str(RESULTS)], check=True)
print('Prueba corta completada para ambos modelos. No usar estas cifras en el artículo.')


## Ejecutar cinco semillas por modelo

Esta celda puede requerir varias sesiones según GPU y disponibilidad. Cada ejecución terminada se conserva en Drive. La descarga inicial de pesos requiere Internet y VGG genera checkpoints grandes. La misma configuración y manifiesto son obligatorios para reutilizar ejecuciones.

Conservar los registros de advertencias. La celda siguiente no accede a las imágenes de prueba.

In [ ]:
import shutil
shutil.copy2('config.json', RESULTS / 'config.json')
shutil.copy2('data/classification.csv', RESULTS / 'classification.csv')
subprocess.run([sys.executable, 'src/run_all.py', '--stage', 'train', '--dataset', str(DATASET), '--results', str(RESULTS)], check=True)


## Evaluar prueba y generar resultados

Ejecutar cuando los diez entrenamientos estén completos. No ajustar modelos a partir de estos resultados. Todas las latencias deben medirse en el mismo modelo de GPU: ejecutar esta celda completa en una sola sesión.

El programa produce CSV por ejecución, matrices de confusión y Grad-CAM de VGG de la primera semilla prefijada.

In [ ]:
subprocess.run([sys.executable, 'src/run_all.py', '--stage', 'evaluate', '--dataset', str(DATASET), '--results', str(RESULTS)], check=True)
subprocess.run([sys.executable, 'src/summarize.py', '--results', str(RESULTS)], check=True)


## Ver resultados y descargar Excel

Con cinco pares, el mínimo p bilateral de la prueba exacta es 0.0625. Informar el resultado sin interpretar falta de significación como igualdad de modelos.

Compartir el Excel y los gráficos para completar el artículo con cifras reales.

In [ ]:
import pandas as pd
from IPython.display import display, Image
EXPERIMENT = RESULTS / 'experiment'
display(pd.read_csv(EXPERIMENT / 'all_runs.csv'))
print((EXPERIMENT / 'statistical_test.json').read_text())
display(Image(filename=str(EXPERIMENT / 'performance_latency.png')))
display(Image(filename=str(EXPERIMENT / 'learning_curves.png')))
files.download(str(EXPERIMENT / 'results.xlsx'))
